# Pda1–TurboID reanalysis: differential enrichment & gene-set testing

Reanalysis of the streptavidin pulldown MS data (Floaty-TurboID / Pda1-TurboID / untagged WT, n=3 each).

**Goal:** replace the uninformative open-GO bar chart with (i) a clean differential-enrichment volcano and
(ii) hypothesis-driven gene-set tests using the **detected proteome as background**.

**Key design decisions (change in the config cell if you disagree):**
- Input is the full MaxQuant `proteinGroups.txt`. Contaminants/reverse hits are removed using the
  flag columns, and LFQ columns are auto-detected as linear-vs-log2 and log2-transformed if linear.
- The `Biotin site` columns are used to flag proteins with a directly MS-localised biotinylation site
  (orthogonal labeling evidence, carried through to the exported hit list).
- Primary "Pda1-proximal proteome" = significantly enriched **Pda1 vs Floaty** AND **Pda1 vs WT**.
  - vs Floaty = spatial specificity over freely diffusing matrix TurboID.
  - vs WT = genuine TurboID-dependent biotinylation (removes the endogenously biotinylated
    carboxylases Acc1/Pyc1/Pyc2/Hfa1, which bind streptavidin in every condition).
- Enrichment **background = every protein passing the valid-value filter**, never the whole genome.

## 1. Setup

Library imports and global settings.

| Library | Role in this notebook |
|---|---|
| `pandas`, `numpy` | Reading the MaxQuant table, intensity transformation, and reshaping |
| `scipy.stats` | Welch's *t*-test for differential enrichment |
| `scipy.stats.hypergeom` | Hypergeometric test for gene-set over-representation |
| `statsmodels.stats.multitest` | Benjamini–Hochberg correction across tests |
| `matplotlib` | Volcano and enrichment figures |
| `re` | Pattern matching for auto-detection of LFQ and Biotin site columns |

The global seed (`np.random.seed(0)`) fixes any stochastic step — notably
imputation — so results are reproducible across runs.

**Note:** only the final g:Profiler cell requires network access. The core
analysis, including the hypergeometric gene-set tests, runs entirely offline
against the detected proteome background.

In [ ]:
# If needed (only the g:Profiler cell at the end requires internet):
# pip install pandas numpy scipy statsmodels matplotlib gprofiler-official

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import re

pd.set_option("display.max_columns", 60)
np.random.seed(0)

## 2. Configuration

**Input.** Tab-separated MaxQuant `proteinGroups.txt`.

**Sample groups.** Three biological replicates per condition:

| Group | Samples | Role |
|---|---|---|
| `Pda1` | `Pda1_1–3` | Pda1-TurboID bait |
| `Floaty` | `Floaty_1–3` | Freely diffusing matrix TurboID — spatial specificity control |
| `WT` | `WT_1–3` | Untagged — TurboID-dependence control |

**Thresholds:**

| Parameter | Value | Meaning |
|---|---|---|
| `FC_CUTOFF` | 1.0 | Log2 fold change (2-fold) |
| `FDR_CUTOFF` | 0.05 | Benjamini–Hochberg adjusted p-value |
| `MIN_VALID` | 2 | Minimum valid values required in the test group of each contrast |
| `IMPUTE_DOWNSHIFT` / `IMPUTE_WIDTH` | 1.8 / 0.3 | Down-shifted normal imputation (Perseus defaults), in column SD units |

**Valid-value filter.** Applied to the *test* group of each contrast only, so a
protein detected in Pda1 but absent from Floaty or WT is retained. This is
deliberate: TurboID-dependent biotinylation is expected to give exactly that
pattern, and requiring valid values in both groups would discard the strongest
candidates.

**Imputation** fills the resulting missing control values from a down-shifted
normal, modelling below-detection absence.

In [ ]:
# ---- point this at your MaxQuant proteinGroups.txt (or the tab-separated export you pasted) ----
DATA_DIR   = "."
INPUT_FILE = f"{DATA_DIR}/ms_results_turboid_pda1.txt"      # tab-separated
SEP = "\t"

# Sample groups -> the suffixes after "LFQ intensity "
GROUPS = {
    "Floaty": ["Floaty_1", "Floaty_2", "Floaty_3"],
    "Pda1":   ["Pda1_1",   "Pda1_2",   "Pda1_3"],
    "WT":     ["WT_1",     "WT_2",     "WT_3"],
}

# Significance thresholds
FC_CUTOFF  = 1.0     # log2 fold change (=2x)
FDR_CUTOFF = 0.05    # BH-adjusted p

# Valid-value filter: require at least this many valid values in the test group of each contrast
MIN_VALID = 2

# Perseus-style imputation of missing control values (downshifted normal)
IMPUTE_WIDTH = 0.3
IMPUTE_DOWNSHIFT = 1.8

OUTDIR = "."

## 3. Load data and detect intensity columns

Reads the MaxQuant table and locates the LFQ intensity columns by their
`LFQ intensity ` prefix, mapping each to its sample suffix.

**Sample check.** Every sample listed in `GROUPS` is asserted to exist in the
file before the analysis proceeds, so a typo or a renamed column fails
immediately with a clear message rather than silently propagating into the
contrasts.

**Scale detection.** Raw MaxQuant LFQ intensities are linear and typically span
10⁶–10⁹, whereas a Perseus export is usually already log2-transformed and falls
in the range 10–30. The maximum observed value is compared against a threshold
of 45, which sits comfortably between the two regimes, and the matrix cell below
log2-transforms only if the values are linear. Zeros are converted to `NaN`
first, so non-detection is treated as missing rather than as a measured value.

In [ ]:
df = pd.read_csv(INPUT_FILE, sep=SEP)
print("rows x cols:", df.shape)

lfq_cols = [c for c in df.columns if c.startswith("LFQ intensity ")]
# map "LFQ intensity Pda1_1" -> "Pda1_1"
sample_of = {c: c.replace("LFQ intensity ", "") for c in lfq_cols}
print("LFQ samples found:", list(sample_of.values()))

# sanity check we have every configured sample
configured = [s for v in GROUPS.values() for s in v]
missing = [s for s in configured if s not in sample_of.values()]
assert not missing, f"configured samples not found in file: {missing}"

In [ ]:
# Report the LFQ value scale. Raw MaxQuant LFQ is linear (~1e6-1e9); a Perseus export is
# often already log2 (~10-30). The matrix cell below auto-detects and log2-transforms if linear.
lfq = df[lfq_cols].apply(pd.to_numeric, errors="coerce").replace(0, np.nan)
vmax = np.nanmax(lfq.values)
print("LFQ value range: %.3g .. %.3g" % (np.nanmin(lfq.values), vmax))
print("-> looks", "LINEAR (will be log2-transformed)" if vmax > 45 else "already LOG2")

## 4. Filtering, gene mapping, and matrix construction

**QC filtering.** MaxQuant flag columns are dropped where present, covering both
the current column names and the older `Contaminant` header. As a fallback for
files exported without flag columns, protein groups whose leading ID carries a
`CON__` or `REV__` prefix are removed by pattern. Each step reports how many
rows it removed.

**Gene mapping.** One symbol per protein group, taken as the first entry of the
semicolon-separated `Gene names` field, falling back to the leading protein ID
where no gene name is annotated.

**Direct-biotinylation flag.** `biotin_site` records whether MaxQuant localised a
biotin modification site on the protein. This is orthogonal evidence: it reflects
a directly observed modification rather than relative abundance, so a hit
supported by both enrichment and a localised site is more strongly supported
than one resting on enrichment alone. The flag is carried through to the
exported hit list, and set to `False` throughout if the file lacks the
`Biotin site` columns.

**Intensity matrix.** LFQ columns are parsed to numeric and indexed by gene
symbol. Zeros encode non-detection and become `NaN`. Linear intensities are
log2-transformed automatically per the scale detection above.

**Helper functions.**

- `valid_filter` — retains proteins with at least `MIN_VALID` measured values in
  a specified group. Applied to the test group of each contrast, so proteins
  absent from a control are kept.
- `impute_downshift` — per-sample down-shifted normal imputation (Perseus
  defaults). The generator is seeded with a fixed default, so imputed values and
  all downstream statistics are reproducible.

In [ ]:
# MaxQuant flag columns - dropped only if present
for flag in ["Reverse", "Potential contaminant", "Only identified by site", "Contaminant"]:
    if flag in df.columns:
        before = len(df)
        df = df[df[flag] != "+"].copy()
        print(f"removed {before-len(df)} rows flagged '{flag}'")

# belt-and-suspenders: drop CON__/REV__ leading protein IDs even if flags are absent
lead = df["Protein IDs"].astype(str).str.split(";").str[0]
keep = ~lead.str.startswith(("CON__", "REV__"))
print(f"removed {int((~keep).sum())} CON__/REV__ rows by ID")
df = df[keep].copy()

# A clean gene symbol per row (first token of 'Gene names'; fall back to protein id)
def first_gene(row):
    g = row.get("Gene names")
    if isinstance(g, str) and g.strip():
        return g.split(";")[0].strip()
    pid = str(row.get("Protein IDs", "")).split(";")[0]
    return pid if pid else None

df["gene"] = df.apply(first_gene, axis=1)

# Direct-biotinylation flag: did MaxQuant localise a biotin modification site on this protein?
def has_biotin(row):
    for col in ["Biotin site IDs", "Biotin site positions"]:
        v = row.get(col)
        if isinstance(v, str) and v.replace(";", "").strip():
            return True
    return False
if any(c.startswith("Biotin site") for c in df.columns):
    df["biotin_site"] = df.apply(has_biotin, axis=1)
    print("proteins with a localised biotin site:", int(df["biotin_site"].sum()))
else:
    df["biotin_site"] = False
    print("no 'Biotin site' columns in this file -> biotin_site flag set False")
biotin_genes = set(df.loc[df["biotin_site"], "gene"])

# Build the intensity matrix indexed by gene; zeros in MaxQuant LFQ mean "not detected"
mat = df[lfq_cols].apply(pd.to_numeric, errors="coerce").copy()
mat.columns = [sample_of[c] for c in lfq_cols]
mat = mat.replace(0, np.nan)

# auto-detect linear vs log2 and transform if needed
if np.nanmax(mat.values) > 45:
    print("LFQ is linear -> applying log2")
    mat = np.log2(mat)
else:
    print("LFQ already log2")

mat.index = df["gene"].values
mat = mat[~mat.index.duplicated(keep="first")]
print("matrix:", mat.shape)
print("missing per sample:\n", mat.isna().sum())

In [ ]:
def valid_filter(matrix, group_samples, min_valid=MIN_VALID):
    "Keep proteins with >= min_valid measured values in the given group."
    return matrix.loc[matrix[group_samples].notna().sum(axis=1) >= min_valid]

def impute_downshift(matrix, width=IMPUTE_WIDTH, downshift=IMPUTE_DOWNSHIFT, seed=0):
    "Perseus-style per-sample downshifted-normal imputation of remaining NaNs."
    rng = np.random.default_rng(seed)
    out = matrix.copy()
    for col in out.columns:
        v = out[col]
        mu = v.mean(skipna=True) - downshift * v.std(skipna=True)
        sd = width * v.std(skipna=True)
        miss = v.isna()
        out.loc[miss, col] = rng.normal(mu, sd, int(miss.sum()))
    return out

## 5. Differential enrichment contrasts

Computes the two contrasts underpinning the Pda1-proximal proteome definition.

**Procedure per contrast.** Proteins are required to have at least `MIN_VALID`
measured values in the *test* group only, so true on/off hits — present with
Pda1, absent from the control — survive the filter. Remaining missing values are
then imputed from a down-shifted normal, and a Welch *t*-test is applied across
the three replicates per group. Log2 fold change is mean(test) − mean(control),
so **positive values indicate enrichment with Pda1**. P-values are corrected by
Benjamini–Hochberg.

**Enrichment call.** q < 0.05 **and** log2FC > 1. One-sided by construction:
proteins depleted relative to a control are never flagged, which is appropriate
for a proximity-labelling enrichment.

| Contrast | What enrichment establishes |
|---|---|
| Pda1 vs Floaty | Spatial specificity — labelled more than by freely diffusing matrix TurboID, so proximity to Pda1 rather than general matrix residence |
| Pda1 vs WT | TurboID dependence — biotinylation requires the enzyme, excluding endogenously biotinylated carboxylases and streptavidin background |

Each contrast is filtered and imputed independently, so the two tables may test
slightly different protein sets. The intersection defines the primary
Pda1-proximal proteome.

In [ ]:
def contrast(matrix, group_A, group_B, label, min_valid=MIN_VALID):
    # Enrichment of A over B. Requires >= min_valid valid values in the test group A,
    # then imputes the (typically absent) control values so true on/off hits are kept.
    # Returns a per-protein table with log2FC = mean(A) - mean(B), p, BH-q.
    sub = matrix[group_A + group_B]
    sub = valid_filter(sub, group_A, min_valid)
    sub = impute_downshift(sub)

    A = sub[group_A].values
    B = sub[group_B].values
    t, p = stats.ttest_ind(A, B, axis=1, equal_var=False)
    res = pd.DataFrame(
        {"log2FC": A.mean(1) - B.mean(1), "t": t, "p": p},
        index=sub.index,
    ).dropna(subset=["p"])
    res["q"] = multipletests(res["p"], method="fdr_bh")[1]
    res["neglog10p"] = -np.log10(res["p"])
    res["sig"] = (res["q"] < FDR_CUTOFF) & (res["log2FC"] > FC_CUTOFF)
    res = res.sort_values("q")
    print(f"{label}: {len(res)} tested, {int(res['sig'].sum())} enriched "
          f"(q<{FDR_CUTOFF}, log2FC>{FC_CUTOFF})")
    return res

res_pf = contrast(mat, GROUPS["Pda1"], GROUPS["Floaty"], "Pda1 vs Floaty")
res_pw = contrast(mat, GROUPS["Pda1"], GROUPS["WT"],     "Pda1 vs WT")
res_pf.head(20)

## 6. Define the Pda1-proximal proteome

**Foreground.** The intersection of proteins significantly enriched in *both*
contrasts — enriched over Floaty-TurboID (spatial specificity) **and** over
untagged WT (TurboID dependence). Requiring both is conservative: a protein
enriched over WT alone could simply be an abundant matrix protein labelled by
any matrix-localised enzyme, while one enriched over Floaty alone could be a
streptavidin-binding background protein. The set sizes for each individual
contrast are reported alongside the intersection, so the contribution of each
filter is visible.

**Background.** The union of proteins tested in either contrast — the detected
proteome, not the genome. All gene-set testing below uses this as the reference
universe, which is the appropriate null for MS data: the genome-wide alternative
would conflate detectability with enrichment and inflate significance for any
term over-represented among abundant proteins.

**Orthogonal support.** Foreground proteins carrying a MaxQuant-localised biotin
modification site are marked with `*`. This is independent evidence of direct
labelling rather than relative abundance, and the fraction of the foreground
carrying a site is a useful quality indicator for the labelling experiment as a
whole.

In [ ]:
fg_pf = set(res_pf.index[res_pf["sig"]])
fg_pw = set(res_pw.index[res_pw["sig"]])

foreground = fg_pf & fg_pw          # primary: specific AND biotinylated
background = set(res_pf.index) | set(res_pw.index)   # the detected/tested proteome

print("enriched vs Floaty :", len(fg_pf))
print("enriched vs WT     :", len(fg_pw))
print("FOREGROUND (both)  :", len(foreground))
print("BACKGROUND         :", len(background))

fg_with_site = foreground & biotin_genes
print(f"\nforeground with a localised biotin site: {len(fg_with_site)}/{len(foreground)}")
print("\nForeground genes (* = direct biotin site):")
print(", ".join(sorted(g + ("*" if g in biotin_genes else "") for g in foreground)))

## 7. Volcano plots — Pda1 vs Floaty

Visualises the spatial-specificity contrast, with points classified by
functional group.

**Classification precedence** (first match wins): PDH complex → mitoribosome
large subunit (54S) → small subunit (37S) → Pda1-proximal foreground → other.
The ribosomal sets are SGD-derived. Note that ribosomal proteins are coloured by
membership in these sets **regardless of whether they were called enriched**, so
coloured points include non-significant proteins; the `Pda1-proximal` class
covers only foreground members not already assigned to one of the earlier
groups.

**Threshold guides.** The vertical line marks the log2FC cutoff. The horizontal
line is placed at the lowest −log10(p) among proteins passing the FDR cutoff,
i.e. the p-value at which BH correction reaches q = 0.05 in this dataset — so
the y-axis shows raw p while the line encodes the adjusted threshold.

**Figure 2C** (second panel) is the publication version: smaller format, no
title, colourblind-safe palette, italic gene labels with leader lines at manual
offsets, and top/right spines removed. `pdf.fonttype = 42` and
`svg.fonttype = 'none'` keep text editable for figure assembly in Affinity
Designer.

**Top-25 inspection panel** (final version) labels the 25 highest-ranked
classified proteins by q-value then log2FC, for internal inspection rather than
publication.

In [ ]:
PDH = {"PDA1", "PDB1", "LAT1", "LPD1", "PDX1"}

def classify(g):
    gu = str(g).upper()
    if gu in PDH: return "PDH"
    name = ""  # tag mito gene-expression hits by name keyword
    if gu in foreground: return "Pda1-proximal"
    return "other"

r = res_pf.copy()
r["cls"] = [classify(g) for g in r.index]
colors = {"other": "#bdbdbd", "Pda1-proximal": "#2c7fb8", "PDH": "#d95f0e"}

fig, ax = plt.subplots(figsize=(6, 5))
for cls in ["other", "Pda1-proximal", "PDH"]:
    sub = r[r["cls"] == cls]
    ax.scatter(sub["log2FC"], sub["neglog10p"], s=18, c=colors[cls],
               edgecolor="none", alpha=0.8, label=cls)

# threshold guides
ax.axvline(FC_CUTOFF, ls="--", c="k", lw=0.6)
qline = r.loc[r["q"] < FDR_CUTOFF, "neglog10p"].min()
if np.isfinite(qline):
    ax.axhline(qline, ls="--", c="k", lw=0.6)

# label PDH subunits
for g in PDH:
    if g in r.index:
        ax.annotate(g, (r.loc[g, "log2FC"], r.loc[g, "neglog10p"]),
                    fontsize=8, fontweight="bold")

ax.set_xlabel("log2 fold change (Pda1 / Floaty)")
ax.set_ylabel("-log10 p")
ax.legend(frameon=False, fontsize=8)
ax.set_title("Pda1-TurboID vs Floaty-TurboID")
fig.tight_layout()
#fig.savefig(f"{OUTDIR}/volcano_Pda1_vs_Floaty.pdf")
#fig.savefig(f"{OUTDIR}/volcano_Pda1_vs_Floaty.png", dpi=200)
plt.show()

# Figure 2C code

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42   # editable fonts in Affinity
matplotlib.rcParams['svg.fonttype'] = 'none'  # text stays as text, not paths

SGD_LARGE_RIBO = {"IMG1", "IMG2", "MHR1", "MNP1", "MRP20", "MRP35", "MRP49", "MRP7",
                  "MRPL1", "MRPL10", "MRPL11", "MRPL13", "MRPL15", "MRPL16", "MRPL17",
                  "MRPL19", "MRPL20", "MRPL22", "MRPL23", "MRPL24", "MRPL25", "MRPL27",
                  "MRPL28", "MRPL3", "MRPL31", "MRPL32", "MRPL33", "MRPL35", "MRPL36",
                  "MRPL37", "MRPL38", "MRPL39", "MRPL4", "MRPL40", "MRPL44", "MRPL49",
                  "MRPL50", "MRPL51", "MRPL6", "MRPL7", "MRPL8", "MRPL9", "MRX14", "PTH4",
                  "RML2", "RTC6", "YML6"}

SGD_SMALL_RIBO = {"EHD3", "FYV4", "MRP1", "MRP10", "MRP13", "MRP17", "MRP2", "MRP21",
                  "MRP4", "MRP51", "MRPS12", "MRPS16", "MRPS17", "MRPS18", "MRPS28",
                  "MRPS35", "MRPS5", "MRPS8", "MRPS9", "NAM9", "PET123", "PPE1", "QRI5",
                  "RSM10", "RSM18", "RSM19", "RSM22", "RSM23", "RSM24", "RSM25", "RSM26",
                  "RSM27", "RSM28", "RSM7", "SWS2", "VAR1"}

PDH = {"PDA1", "PDB1", "LAT1", "LPD1", "PDX1"}

def classify(g):
    gu = str(g).upper()
    if gu in PDH:                return "PDH"
    if gu in SGD_LARGE_RIBO:    return "LSU"
    if gu in SGD_SMALL_RIBO:    return "SSU"
    if gu in foreground:         return "proximal"
    return "other"

r = res_pf.copy()
r["cls"] = [classify(g) for g in r.index]

# Colors: keep colorblind-friendly
# grey / light blue / red / teal / orange
colors = {
    "other":    "#c0c0c0",
    "proximal": "#9ecae1",
    "LSU":      "#d62728",   # red
    "SSU":      "#2ca02c",   # green
    "PDH":      "#e6801a",   # orange
}

# Point sizes: make ribosomal and PDH dots slightly larger to pop
sizes = {
    "other":    14,
    "proximal": 16,
    "LSU":      22,
    "SSU":      22,
    "PDH":      26,
}

# zorder: background at bottom, PDH on top
zorder = {"other": 1, "proximal": 2, "LSU": 3, "SSU": 3, "PDH": 4}

fig, ax = plt.subplots(figsize=(5, 4.5))

for cls in ["other", "proximal", "LSU", "SSU", "PDH"]:
    sub = r[r["cls"] == cls]
    ax.scatter(sub["log2FC"], sub["neglog10p"],
               s=sizes[cls], c=colors[cls],
               edgecolors="none", alpha=0.9,
               zorder=zorder[cls],
               label={"other": "other",
                      "proximal": "Pda1-proximal",
                      "LSU": "Mito LSU (54S)",
                      "SSU": "Mito SSU (37S)",
                      "PDH": "PDH complex"}[cls])

# Threshold guide lines
ax.axvline(FC_CUTOFF, ls="--", c="#555555", lw=0.7, zorder=0)
qline = r.loc[r["q"] < FDR_CUTOFF, "neglog10p"].min()
if np.isfinite(qline):
    ax.axhline(qline, ls="--", c="#555555", lw=0.7, zorder=0)

# Label PDH subunits only — adjust offsets in Affinity if they overlap
label_offsets = {
    "PDA1": (0.15,  0.05),
    "PDB1": (0.15,  0.05),
    "LAT1": (0.15,  0.05),
    "LPD1": (0.15, -0.15),
    "PDX1": (0.15,  0.05),
}
for g in PDH:
    if g in r.index:
        dx, dy = label_offsets.get(g, (0.15, 0.05))
        ax.annotate(g,
                    xy=(r.loc[g, "log2FC"], r.loc[g, "neglog10p"]),
                    xytext=(r.loc[g, "log2FC"] + dx,
                            r.loc[g, "neglog10p"] + dy),
                    fontsize=7, fontstyle="italic",
                    arrowprops=dict(arrowstyle="-", color="#888888", lw=0.5),
                    zorder=5)

ax.set_xlabel(r"$\log_2$ fold change (Pda1--TurboID / Floaty--TurboID)", fontsize=9)
ax.set_ylabel(r"$-\log_{10}$ $p$", fontsize=9)
ax.tick_params(labelsize=8)
ax.legend(frameon=False, fontsize=7, loc="upper left",
          markerscale=1.2, handletextpad=0.4)
ax.set_title("")   # no title for paper figure

# remove top and right spines (cleaner for publication)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

# SVG for Affinity Designer, PDF as backup
#fig.savefig(f"{OUTDIR}/figure2C_volcano.svg", format="svg", dpi=300)
#fig.savefig(f"{OUTDIR}/figure2C_volcano.pdf", format="pdf", dpi=300)
plt.show()
print("Saved figure2C_volcano.svg and .pdf")

In [ ]:
# SGD-derived ribosomal subunit sets (from the enrichment analysis)
SGD_LARGE_RIBO = {"IMG1", "IMG2", "MHR1", "MNP1", "MRP20", "MRP35", "MRP49", "MRP7",
                  "MRPL1", "MRPL10", "MRPL11", "MRPL13", "MRPL15", "MRPL16", "MRPL17",
                  "MRPL19", "MRPL20", "MRPL22", "MRPL23", "MRPL24", "MRPL25", "MRPL27",
                  "MRPL28", "MRPL3", "MRPL31", "MRPL32", "MRPL33", "MRPL35", "MRPL36",
                  "MRPL37", "MRPL38", "MRPL39", "MRPL4", "MRPL40", "MRPL44", "MRPL49",
                  "MRPL50", "MRPL51", "MRPL6", "MRPL7", "MRPL8", "MRPL9", "MRX14", "PTH4",
                  "RML2", "RTC6", "YML6"}

SGD_SMALL_RIBO = {"EHD3", "FYV4", "MRP1", "MRP10", "MRP13", "MRP17", "MRP2", "MRP21",
                  "MRP4", "MRP51", "MRPS12", "MRPS16", "MRPS17", "MRPS18", "MRPS28",
                  "MRPS35", "MRPS5", "MRPS8", "MRPS9", "NAM9", "PET123", "PPE1", "QRI5",
                  "RSM10", "RSM18", "RSM19", "RSM22", "RSM23", "RSM24", "RSM25", "RSM26",
                  "RSM27", "RSM28", "RSM7", "SWS2", "VAR1"}

PDH = {"PDA1", "PDB1", "LAT1", "LPD1", "PDX1"}

def classify(g):
    gu = str(g).upper()
    if gu in PDH:
        return "PDH"
    if gu in SGD_LARGE_RIBO:
        return "Large ribosomal subunit"
    if gu in SGD_SMALL_RIBO:
        return "Small ribosomal subunit"
    if gu in foreground:
        return "Pda1-proximal"
    return "other"

r = res_pf.copy()
r["cls"] = [classify(g) for g in r.index]

colors = {"other": "#bdbdbd", "Pda1-proximal": "#2c7fb8",
          "Large ribosomal subunit": "#e41a1c", "Small ribosomal subunit": "#4daf4a",
          "PDH": "#d95f0e"}

fig, ax = plt.subplots(figsize=(6, 5))
# plot in order: background first, then hits, then PDH on top
for cls in ["other", "Pda1-proximal", "Large ribosomal subunit", "Small ribosomal subunit", "PDH"]:
    sub = r[r["cls"] == cls]
    ax.scatter(sub["log2FC"], sub["neglog10p"], s=18, c=colors[cls],
               edgecolor="none", alpha=0.8, label=cls)

# threshold guides
ax.axvline(FC_CUTOFF, ls="--", c="k", lw=0.6)
qline = r.loc[r["q"] < FDR_CUTOFF, "neglog10p"].min()
if np.isfinite(qline):
    ax.axhline(qline, ls="--", c="k", lw=0.6)

# label only PDH subunits
for g in PDH:
    if g in r.index:
        ax.annotate(g, (r.loc[g, "log2FC"], r.loc[g, "neglog10p"]),
                    fontsize=8, fontweight="bold")

ax.set_xlabel("log2 fold change (Pda1 / Floaty)")
ax.set_ylabel("-log10 p")
ax.legend(frameon=False, fontsize=8)
ax.set_title("Pda1-TurboID vs Floaty-TurboID")
fig.tight_layout()
#fig.savefig(f"{OUTDIR}/volcano_Pda1_vs_Floaty.pdf")
#fig.savefig(f"{OUTDIR}/volcano_Pda1_vs_Floaty.png", dpi=200)
plt.show()

In [ ]:
# Inspection plot: label the best-enriched 25 hits (ranked by q-value, then log2FC)

# SGD ribosomal sets (reuse from previous cell)
SGD_LARGE_RIBO = {"IMG1", "IMG2", "MHR1", "MNP1", "MRP20", "MRP35", "MRP49", "MRP7",
                  "MRPL1", "MRPL10", "MRPL11", "MRPL13", "MRPL15", "MRPL16", "MRPL17",
                  "MRPL19", "MRPL20", "MRPL22", "MRPL23", "MRPL24", "MRPL25", "MRPL27",
                  "MRPL28", "MRPL3", "MRPL31", "MRPL32", "MRPL33", "MRPL35", "MRPL36",
                  "MRPL37", "MRPL38", "MRPL39", "MRPL4", "MRPL40", "MRPL44", "MRPL49",
                  "MRPL50", "MRPL51", "MRPL6", "MRPL7", "MRPL8", "MRPL9", "MRX14", "PTH4",
                  "RML2", "RTC6", "YML6"}

SGD_SMALL_RIBO = {"EHD3", "FYV4", "MRP1", "MRP10", "MRP13", "MRP17", "MRP2", "MRP21",
                  "MRP4", "MRP51", "MRPS12", "MRPS16", "MRPS17", "MRPS18", "MRPS28",
                  "MRPS35", "MRPS5", "MRPS8", "MRPS9", "NAM9", "PET123", "PPE1", "QRI5",
                  "RSM10", "RSM18", "RSM19", "RSM22", "RSM23", "RSM24", "RSM25", "RSM26",
                  "RSM27", "RSM28", "RSM7", "SWS2", "VAR1"}

PDH = {"PDA1", "PDB1", "LAT1", "LPD1", "PDX1"}

def classify(g):
    gu = str(g).upper()
    if gu in PDH:
        return "PDH"
    if gu in SGD_LARGE_RIBO:
        return "Large ribosomal subunit"
    if gu in SGD_SMALL_RIBO:
        return "Small ribosomal subunit"
    if gu in foreground:
        return "Pda1-proximal"
    return "other"

r = res_pf.copy()
r["cls"] = [classify(g) for g in r.index]

colors = {"other": "#bdbdbd", "Pda1-proximal": "#2c7fb8",
          "Large ribosomal subunit": "#e41a1c", "Small ribosomal subunit": "#4daf4a",
          "PDH": "#d95f0e"}

# Select top 25 enriched hits: filter to foreground, rank by q-value then log2FC
top25 = r[r["cls"] != "other"].sort_values(["q", "log2FC"], ascending=[True, False]).head(25)
print(f"Top 25 enriched hits (ranked by q-value):")
print(top25[["log2FC", "q", "cls"]].to_string())

fig, ax = plt.subplots(figsize=(9, 7))

# plot all points in background order
for cls in ["other", "Pda1-proximal", "Large ribosomal subunit", "Small ribosomal subunit", "PDH"]:
    sub = r[r["cls"] == cls]
    ax.scatter(sub["log2FC"], sub["neglog10p"], s=18, c=colors[cls],
               edgecolor="none", alpha=0.8, label=cls)

# threshold guides
ax.axvline(FC_CUTOFF, ls="--", c="k", lw=0.6, alpha=0.5)
qline = r.loc[r["q"] < FDR_CUTOFF, "neglog10p"].min()
if np.isfinite(qline):
    ax.axhline(qline, ls="--", c="k", lw=0.6, alpha=0.5)

# label the top 25
for g in top25.index:
    ax.annotate(g, (r.loc[g, "log2FC"], r.loc[g, "neglog10p"]),
                fontsize=7, alpha=0.75, ha="right")

ax.set_xlabel("log2 fold change (Pda1 / Floaty)")
ax.set_ylabel("-log10 p")
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("Pda1-TurboID vs Floaty-TurboID (top 25 enriched, labeled for inspection)")
fig.tight_layout()
#fig.savefig(f"{OUTDIR}/volcano_Pda1_vs_Floaty_top25_labeled.pdf")
#fig.savefig(f"{OUTDIR}/volcano_Pda1_vs_Floaty_top25_labeled.png", dpi=200)
plt.show()

## 8. Export enrichment table

Writes the Pda1 vs Floaty contrast to Excel, sorted by adjusted p-value and then
by decreasing fold change. Columns give the gene symbol, log2 fold change,
BH-adjusted q-value, −log10 raw p, and the functional category used for
colouring in the volcano.

Fold changes are mean(Pda1) − mean(Floaty), so positive values indicate
enrichment with Pda1-TurboID over freely diffusing matrix TurboID.

In [ ]:
# ---- export the Pda1-TurboID vs Floaty enrichment table (supplementary) ----
turboid_out = (r.reset_index()
               .rename(columns={"index": "gene",
                                "log2FC": "log2FC_Pda1_vs_Floaty",
                                "q": "q_value",
                                "cls": "category"})
               [["gene", "log2FC_Pda1_vs_Floaty", "q_value", "neglog10p", "category"]]
               .sort_values(["q_value", "log2FC_Pda1_vs_Floaty"],
                            ascending=[True, False]))

turboid_out.to_excel(f"{OUTDIR}/pda1_turboid_enrichment.xlsx",
                     index=False, sheet_name="Pda1-TurboID")
print(f"wrote pda1_turboid_enrichment.xlsx  ({len(turboid_out)} proteins)")

## 9. Curated gene-set enrichment

Hypothesis-driven testing of specific functional sets against the detected
proteome, replacing an open GO query. Sets are SGD-derived (YeastMine,
GO-annotated) rather than assembled by name pattern, so set sizes reflect
annotation rather than what protein-name strings happen to match.

| Set | Role in the analysis |
|---|---|
| Mito large ribosomal subunit (GO:0005762) | Test |
| Mito small ribosomal subunit (GO:0005763) | Test |
| Mito translation factors, non-ribosomal | Derived: translation minus both ribosomal subunits. Tests whether the non-ribosomal machinery is enriched independently of the ribosome itself |
| Mito translation, all | Superset including ribosomes |
| mtDNA nucleoid | Test |
| TCA cycle | Comparison set |
| OXPHOS | Comparison set |

**Coverage is reported before testing.** Sets with few members detected in the
background have little power regardless of the underlying biology, so coverage
is printed so that a null result can be read as low power rather than absence
of enrichment.

**Test.** One-sided hypergeometric over-representation of each set within the
foreground, drawn against the detected proteome as the universe. Each set is
first intersected with the background, so undetected members do not count
against it. Fold enrichment is the observed foreground fraction over the
background fraction. P-values are BH-corrected across the sets tested.

### Figure — set enrichment

Horizontal bars of −log10 q, with comparison sets in grey and test sets in
blue, a dashed line at q = 0.05, and each bar annotated with the
foreground/set-size ratio so the reader sees the counts behind each q-value.

In [ ]:
# Canonical gene sets from SGD (YeastMine, GO-annotated; see scratch cells at end).
# Replacing the earlier regex/hand-curated lists with these makes the "set sizes"
# reflect the true biology rather than what the protein-name strings happen to match.

SGD_LARGE_RIBO = {
    "IMG1","IMG2","MHR1","MNP1","MRP20","MRP35","MRP49","MRP7",
    "MRPL1","MRPL10","MRPL11","MRPL13","MRPL15","MRPL16","MRPL17","MRPL19",
    "MRPL20","MRPL22","MRPL23","MRPL24","MRPL25","MRPL27","MRPL28","MRPL3",
    "MRPL31","MRPL32","MRPL33","MRPL35","MRPL36","MRPL37","MRPL38","MRPL39",
    "MRPL4","MRPL40","MRPL44","MRPL49","MRPL50","MRPL51","MRPL6","MRPL7",
    "MRPL8","MRPL9","MRX14","PTH4","RML2","RTC6","YML6",
}  # SGD GO:0005762, 47 genes

SGD_SMALL_RIBO = {
    "EHD3","FYV4","MRP1","MRP10","MRP13","MRP17","MRP2","MRP21","MRP4","MRP51",
    "MRPS12","MRPS16","MRPS17","MRPS18","MRPS28","MRPS35","MRPS5","MRPS8","MRPS9",
    "NAM9","PET123","PPE1","QRI5","RSM10","RSM18","RSM19","RSM22","RSM23","RSM24",
    "RSM25","RSM26","RSM27","RSM28","RSM7","SWS2","VAR1",
}  # SGD GO:0005763, 36 genes

SGD_NUCLEOID = {
    "ABF2","ACO1","ALD4","ATP1","CHA1","ECM10","HSP60","IDH1","IDP1","ILV5","ILV6",
    "KGD1","KGD2","LPD1","LSC1","MGM101","MNP1","PDA1","PDB1","RIM1","RPO41","SLS1",
    "SSC1","YHM2",
}  # SGD mitochondrial nucleoid, 25 genes -- note: PDA1/PDB1 themselves annotated here

SGD_TRANSLATION = {
    "DPC29","EHD3","FYV4","GTF1","HER2","HTS1","IFM1","IMG1","IMG2","ISM1","MEF1",
    "MEF2","MHR1","MMF1","MNP1","MRF1","MRP1","MRP10","MRP13","MRP17","MRP2","MRP20",
    "MRP21","MRP35","MRP4","MRP49","MRP51","MRP7","MRPL1","MRPL10","MRPL11","MRPL13",
    "MRPL15","MRPL16","MRPL17","MRPL19","MRPL20","MRPL22","MRPL23","MRPL24","MRPL25",
    "MRPL27","MRPL28","MRPL3","MRPL31","MRPL32","MRPL33","MRPL35","MRPL36","MRPL37",
    "MRPL38","MRPL39","MRPL4","MRPL40","MRPL44","MRPL49","MRPL50","MRPL51","MRPL6",
    "MRPL7","MRPL8","MRPL9","MRPS12","MRPS16","MRPS17","MRPS18","MRPS28","MRPS35",
    "MRPS5","MRPS8","MRPS9","MRX14","MSE1","MSK1","MSR1","MTF2","NAM2","NAM9","PET112",
    "PET123","PTH1","PTH4","QRI5","RML2","RRF1","RSM10","RSM18","RSM19","RSM22","RSM23",
    "RSM24","RSM25","RSM26","RSM27","RSM28","RSM7","RSO55","RTC6","SLS1","SOV1","SWS2",
    "TUF1","VAR1","YML6",
}  # SGD mitochondrial translation, 106 genes (superset including ribosomes)

SGD_TCA = {
    "ACO1","ACO2","CIT1","CIT2","CIT3","FUM1","IDH1","IDH2","IDP1","IDP2","IDP3",
    "KGD1","KGD2","KGD4","LPD1","LSC1","LSC2","MDH1","MDH2","MDH3","SDH1","SDH2",
    "SDH3","SDH4","SDH5","SDH9","SHH3","SHH4",
}  # 29 genes -- negative control

# Derived: non-ribosomal translation machinery (factors, synthetases, RNA-handling)
# Tests whether the *non-ribosomal* part of translation is enriched, independently
# of the ribosome itself. If only this one came up flat while the ribosomes are
# strong, the signal is specifically the ribosome.
SGD_TRANS_FACTORS = SGD_TRANSLATION - SGD_LARGE_RIBO - SGD_SMALL_RIBO


SGD_OXPHOS = {'ATP1', 'ATP14', 'ATP15', 'ATP16', 'ATP17', 'ATP18', 'ATP19', 'ATP2', 'ATP20',
           'ATP3', 'ATP4', 'ATP5', 'ATP6', 'ATP7', 'ATP8', 'COB', 'COR1', 'COX1', 'COX12',
           'COX13', 'COX2', 'COX26', 'COX3', 'COX4', 'COX5A', 'COX5B', 'COX6', 'COX7', 'COX8',
           'COX9', 'CYT1', 'MTC3', 'OLI1', 'QCR10', 'QCR2', 'QCR6', 'QCR7', 'QCR8', 'QCR9',
           'RCF1', 'RCF2', 'RIP1', 'SDH1', 'SDH2', 'SDH3', 'SDH4', 'SHH3', 'SHH4', 'TIM11',
           'YJL045W', 'YOR020W-A'}


curated = {
    "Mito large ribosomal subunit (SGD)":     SGD_LARGE_RIBO,
    "Mito small ribosomal subunit (SGD)":     SGD_SMALL_RIBO,
    "Mito translation factors, non-ribo":     SGD_TRANS_FACTORS,
    "Mito translation, all (SGD)":            SGD_TRANSLATION,
    "mtDNA nucleoid (SGD)":                   SGD_NUCLEOID,
    "TCA cycle (SGD, comparison)":            SGD_TCA,
    "OXPHOS (SGD, comparison)":               SGD_OXPHOS,
}

# How much of each set did the MS actually detect? Sets with low background
# coverage have weak statistical power -- worth seeing before running the test.
print("Coverage of each set in the detected proteome (background):")
for name, gs in curated.items():
    print(f"  {name:40s} {len(gs & background):3d} / {len(gs):3d}")
print()

def hypergeom_test(fg, bg, gene_set):
    gene_set = gene_set & bg
    N, K, n = len(bg), len(gene_set), len(fg)
    k = len(gene_set & fg)
    p = hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
    fold = (k / n) / (K / N) if (K and n and k) else float("nan")
    return dict(set_size=K, in_fg=k, fold_enrichment=fold, p=p)

rows = []
for name, gs in curated.items():
    if len(gs & background) == 0:
        continue
    rec = hypergeom_test(foreground, background, gs)
    rec["set"] = name
    rows.append(rec)

enr = pd.DataFrame(rows).set_index("set")
enr["q"] = multipletests(enr["p"], method="fdr_bh")[1]
enr["neglog10q"] = -np.log10(enr["q"])
enr = enr.sort_values("q")
enr


In [ ]:
# ---- export the functional-set enrichment table (Fig 2D data, supplementary) ----
enr_out = (enr.reset_index()
           .rename(columns={"set": "gene_set",
                            "set_size": "set_size_detected",
                            "in_fg": "in_Pda1_proximal",
                            "fold_enrichment": "fold_enrichment",
                            "p": "p_value",
                            "q": "q_value",
                            "neglog10q": "neglog10_q"})
           [["gene_set", "set_size_detected", "in_Pda1_proximal",
             "fold_enrichment", "p_value", "q_value"]])

enr_out.to_excel(f"{OUTDIR}/pda1_turboid_set_enrichment.xlsx",
                 index=False, sheet_name="Set enrichment (Fig 2D)")
print(f"wrote pda1_turboid_set_enrichment.xlsx  ({len(enr_out)} sets)")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
e = enr.sort_values("neglog10q")
bars = ax.barh(e.index, e["neglog10q"],
               color=["#2c7fb8" if "comparison" not in s else "#9e9e9e" for s in e.index])
ax.axvline(-np.log10(0.05), ls="--", c="k", lw=0.7)
ax.set_xlabel("-log10 BH q  (hypergeometric vs detected proteome)")
for i, (s, row) in enumerate(e.iterrows()):
    ax.text(row["neglog10q"] + 0.05, i, f"{int(row['in_fg'])}/{int(row['set_size'])}",
            va="center", fontsize=8)
ax.set_title("Curated gene-set enrichment of the Pda1-proximal proteome")
fig.tight_layout()
#fig.savefig(f"{OUTDIR}/curated_set_enrichment.pdf")
#fig.savefig(f"{OUTDIR}/curated_set_enrichment.png", dpi=200)
plt.show()
